In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from utils import get_stock_data

In [ ]:
# Pull 2024 daily data for 浙江震元, forward-adjusted prices
zy = get_stock_data('sh.600114', '2024-01-01', '2024-12-31')
zy.head()

In [ ]:
zy['returns'] = zy['close'].pct_change()

In [ ]:
zy.to_csv('data/sh600114_2024.csv')

In [ ]:
print(f"Rows: {len(zy)}")
print(f"Date range: {zy.index.min()} to {zy.index.max()}")
print(f"Non-null returns: {zy['returns'].notna().sum()}")
print(zy[['close', 'returns']].head())

In [ ]:
returns = zy['returns'].dropna()

paired = pd.DataFrame({
    'yesterday': returns.shift(1),
    'today': returns
}).dropna()

In [ ]:
print(f"Paired rows: {len(paired)}")
print(paired.head())

In [ ]:
group_up = paired.loc[paired['yesterday'] > 0, 'today']
group_down = paired.loc[paired['yesterday'] < 0, 'today']

print(f"Group A (day after up): n = {len(group_up)}, mean = {group_up.mean():.5f}, std = {group_up.std():.5f}")
print(f"Group B (day after down): n = {len(group_down)}, mean = {group_down.mean():.5f}, std = {group_down.std():.5f}")

In [ ]:
t_stat, p_value = stats.ttest_ind(group_down, group_up, equal_var=False)
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

In [ ]:
# The pooled set of all "today" returns, regardless of which group they ended up in
all_today = paired['today'].values
n_a = len(group_up)  # 121
n_b = len(group_down)  # 118

# Observed difference
observed_diff = group_down.mean() - group_up.mean()

# Permutation loop
rng = np.random.default_rng(42)
n_perms = 10000
null_diffs = np.empty(n_perms)

for i in range(n_perms):
    shuffled = rng.permutation(all_today)
    fake_a = shuffled[:n_a]
    fake_b = shuffled[n_a:n_a + n_b]
    null_diffs[i] = fake_b.mean() - fake_a.mean()

# Two-tailed p-value: fraction of null draws at least as extreme as observed
p_perm = np.mean(np.abs(null_diffs) >= np.abs(observed_diff))
print(f"Observed difference: {observed_diff:.5f}")
print(f"Permutation p-value: {p_perm:.4f}")

In [1]:
import pandas as pd
import numpy as np
import os
import time
from utils import get_stock_data

# ─── Setup ─────────────────────────────────────────
os.makedirs('data/prices', exist_ok=True)

start_date = '2024-01-01'
end_date = '2026-04-22'

# ─── Load codes from existing CSV ─────────────────
codes_df = pd.read_csv('data/zz1000_all_codes.csv')
all_codes = codes_df['code'].tolist()
print(f"Loaded {len(all_codes)} codes from CSV")
print(f"First 5: {all_codes[:5]}")

# ─── Pull loop with cache-aware restart ───────────
basket_data = {}
failed = []

for i, code in enumerate(all_codes):
    cache_path = f"data/prices/{code.replace('.', '_')}.csv"
    
    # Skip if already cached
    if os.path.exists(cache_path):
        try:
            df = pd.read_csv(cache_path, parse_dates=['date']).set_index('date')
            basket_data[code] = df
            if (i + 1) % 100 == 0:
                print(f"[{i+1}/{len(all_codes)}] cached, {len(basket_data)} loaded, {len(failed)} failed")
            continue
        except Exception:
            pass  # re-pull if cache is corrupted
    
    try:
        df = get_stock_data(code, start_date, end_date)
        df['returns'] = df['close'].pct_change()
        basket_data[code] = df
        df.to_csv(cache_path)
        if (i + 1) % 100 == 0:
            print(f"[{i+1}/{len(all_codes)}] fresh pull, {len(basket_data)} loaded, {len(failed)} failed")
    except Exception as e:
        failed.append((code, str(e)))
    
    time.sleep(0.1)

print(f"\n=== Pull complete ===")
print(f"Successfully loaded: {len(basket_data)}")
print(f"Failed: {len(failed)}")
if failed and len(failed) < 30:
    print("\nFailures:")
    for code, err in failed:
        print(f"  {code}: {err}")
elif failed:
    print(f"\n({len(failed)} failures, check the 'failed' list directly)")

Loaded 1000 codes from CSV
First 5: ['sz.000012', 'sz.000019', 'sz.000028', 'sz.000029', 'sz.000030']
login success!
logout success!
login success!
logout success!
login success!
logout success!
login success!
logout success!
login success!
logout success!
login success!
logout success!
login success!
logout success!
login success!
logout success!
login success!
logout success!
login success!
logout success!
login success!
logout success!
login success!
logout success!
login success!
logout success!
login success!
logout success!
login success!
logout success!
login success!
logout success!
[100/1000] fresh pull, 100 loaded, 0 failed
login success!
logout success!
login success!
logout success!
login success!
logout success!
login success!
logout success!
login success!
logout success!
login success!
logout success!
login success!
logout success!
login success!
logout success!
login success!
logout success!
login success!
logout success!
login success!
logout success!
login success!
lo

In [2]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy import stats

# ─── Reload all cached stock data ─────────────────
cache_dir = 'data/prices'
basket_data = {}

for fname in os.listdir(cache_dir):
    if not fname.endswith('.csv'):
        continue
    # Skip the codes file if it's in here
    if 'code' in fname.lower():
        continue
    
    code = fname.replace('.csv', '').replace('_', '.', 1)  # sh_600114 -> sh.600114
    try:
        df = pd.read_csv(f'{cache_dir}/{fname}', parse_dates=['date']).set_index('date')
        basket_data[code] = df
    except Exception as e:
        print(f"Failed to load {fname}: {e}")

print(f"Loaded {len(basket_data)} stock DataFrames from cache")

Loaded 999 stock DataFrames from cache


In [3]:
# ─── Build wide-format returns matrix ─────────────
returns_wide = pd.DataFrame({
    code: df['returns'] for code, df in basket_data.items()
})

print(f"Returns matrix shape: {returns_wide.shape}")
print(f"Date range: {returns_wide.index.min().date()} to {returns_wide.index.max().date()}")
print(f"Stocks with data on first date: {returns_wide.iloc[0].notna().sum()}")
print(f"Stocks with data on last date: {returns_wide.iloc[-1].notna().sum()}")

Returns matrix shape: (556, 999)
Date range: 2024-01-02 to 2026-04-22
Stocks with data on first date: 0
Stocks with data on last date: 999


In [4]:
# ─── Equal-weighted basket return per day ─────────
basket_returns = returns_wide.mean(axis=1, skipna=True)
basket_returns = basket_returns.dropna()

print(f"\nBasket return series: {len(basket_returns)} days")
print(f"Mean daily return: {basket_returns.mean():.5f} ({basket_returns.mean()*242:.3f} annualized)")
print(f"Daily std: {basket_returns.std():.5f} ({basket_returns.std()*np.sqrt(242):.3f} annualized)")
print(f"Skewness: {basket_returns.skew():.3f}")
print(f"Excess kurtosis: {basket_returns.kurtosis():.3f}")

# Save
basket_returns.to_csv('data/zz1000_basket_returns_2024_2026.csv', header=['returns'])


Basket return series: 555 days
Mean daily return: 0.00103 (0.248 annualized)
Daily std: 0.01758 (0.273 annualized)
Skewness: -0.088
Excess kurtosis: 8.749


In [5]:
# Check what's happening on the first few dates
print("First 3 dates, non-NaN count per date:")
for date in returns_wide.index[:3]:
    print(f"  {date.date()}: {returns_wide.loc[date].notna().sum()} stocks with data")

print("\nSecond-date stocks (should be ~930 if my theory is right):")
print(returns_wide.iloc[1].notna().sum())

First 3 dates, non-NaN count per date:
  2024-01-02: 0 stocks with data
  2024-01-03: 969 stocks with data
  2024-01-04: 970 stocks with data

Second-date stocks (should be ~930 if my theory is right):
969


In [6]:
print(f"Basket first date: {basket_returns.index.min().date()}")
print(f"Basket last date: {basket_returns.index.max().date()}")
print(f"Expected ~556 trading days; got {len(basket_returns)}")

Basket first date: 2024-01-03
Basket last date: 2026-04-22
Expected ~556 trading days; got 555


In [8]:
# Skip the first row (pct_change artifact), then check for complete history
full_history = returns_wide.iloc[1:].notna().all(axis=0).sum()
print(f"Stocks with complete data from Jan 3 onward: {full_history}")

# Also check: stocks with at most N missing days (for various N)
for threshold in [0, 5, 10, 25, 50, 100]:
    count = (returns_wide.iloc[1:].isna().sum(axis=0) <= threshold).sum()
    print(f"Stocks with ≤ {threshold} missing days: {count}")

Stocks with complete data from Jan 3 onward: 969
Stocks with ≤ 0 missing days: 969
Stocks with ≤ 5 missing days: 970
Stocks with ≤ 10 missing days: 970
Stocks with ≤ 25 missing days: 971
Stocks with ≤ 50 missing days: 973
Stocks with ≤ 100 missing days: 974


In [9]:
# ─── Build the pair DataFrame ─────────────────────
paired = pd.DataFrame({
    'yesterday': basket_returns.shift(1),
    'today': basket_returns
}).dropna()

# Split
group_up = paired.loc[paired['yesterday'] > 0, 'today']
group_down = paired.loc[paired['yesterday'] < 0, 'today']

print(f"Group A (day after up):   n = {len(group_up)}, mean = {group_up.mean():.5f}, std = {group_up.std():.5f}")
print(f"Group B (day after down): n = {len(group_down)}, mean = {group_down.mean():.5f}, std = {group_down.std():.5f}")
print(f"Flat-yesterday days excluded: {len(paired) - len(group_up) - len(group_down)}")

Group A (day after up):   n = 303, mean = 0.00220, std = 0.01665
Group B (day after down): n = 251, mean = -0.00037, std = 0.01861
Flat-yesterday days excluded: 0


In [10]:
t_stat, p_value = stats.ttest_ind(group_up, group_down, equal_var=False)
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

t-statistic: 1.6949
p-value: 0.0907


In [11]:
# Permutation test
all_today = paired['today'].values
n_a = len(group_up)
observed_diff = group_up.mean() - group_down.mean()

rng = np.random.default_rng(42)
n_perms = 10000
null_diffs = np.empty(n_perms)

for i in range(n_perms):
    shuffled = rng.permutation(all_today)
    null_diffs[i] = shuffled[:n_a].mean() - shuffled[n_a:].mean()

p_perm = np.mean(np.abs(null_diffs) >= np.abs(observed_diff))
print(f"Observed difference: {observed_diff:.5f}")
print(f"Permutation p-value: {p_perm:.4f}")

Observed difference: 0.00257
Permutation p-value: 0.0859


In [12]:
# Define stimulus window
stimulus_start = pd.Timestamp('2024-09-24')
stimulus_end = pd.Timestamp('2024-10-10')

# Exclude stimulus days from the pair DataFrame
ex_stim = paired.loc[
    (paired.index < stimulus_start) | (paired.index > stimulus_end)
].copy()

print(f"Full sample: {len(paired)} paired days")
print(f"Ex-stimulus sample: {len(ex_stim)} paired days")
print(f"Excluded: {len(paired) - len(ex_stim)} days")

# Re-split
group_up_ex = ex_stim.loc[ex_stim['yesterday'] > 0, 'today']
group_down_ex = ex_stim.loc[ex_stim['yesterday'] < 0, 'today']

print(f"\nGroup A (day after up), ex-stim:   n = {len(group_up_ex)}, mean = {group_up_ex.mean():.5f}, std = {group_up_ex.std():.5f}")
print(f"Group B (day after down), ex-stim: n = {len(group_down_ex)}, mean = {group_down_ex.mean():.5f}, std = {group_down_ex.std():.5f}")

# Run the test
t_ex, p_ex = stats.ttest_ind(group_up_ex, group_down_ex, equal_var=False)
observed_diff_ex = group_up_ex.mean() - group_down_ex.mean()
print(f"\nEx-stimulus: observed gap = {observed_diff_ex:.5f}")
print(f"Ex-stimulus: t = {t_ex:.4f}, p = {p_ex:.4f}")

Full sample: 554 paired days
Ex-stimulus sample: 546 paired days
Excluded: 8 days

Group A (day after up), ex-stim:   n = 296, mean = 0.00136, std = 0.01305
Group B (day after down), ex-stim: n = 250, mean = -0.00035, std = 0.01864

Ex-stimulus: observed gap = 0.00171
Ex-stimulus: t = 1.2169, p = 0.2243
